In [0]:
!pip install lightgbm

In [0]:
import os
import re
import gc
import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

import pyarrow as pa
import pyarrow.dataset as ds

import lightgbm as lgb
import matplotlib.pyplot as plt
import psutil

In [0]:
# ============================================================
# Input paths
# ============================================================

DATA_PATH = (
    "/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
    "feature_selection_mdb_25k_final_parquet"
)

OUTPUT_DIR = (
    "/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/v2/"
)

MODEL_PATH = "lightgbm_nbrx_model_wo_segment_new_7.txt"

FEATURE_MAPPING_PATH = os.path.join(OUTPUT_DIR, "feature_alias_mapping_wo_segment.csv")

booster = lgb.Booster(model_file=MODEL_PATH)

feature_mapping_df = pd.read_csv(FEATURE_MAPPING_PATH)

LOCAL_OUTPUT_DIR = "/tmp/lgbm_shap_dependency_outputs"
Path(LOCAL_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ============================================================
# Target column
# ============================================================

TARGET_COL = "TOTAL_NBRX_C4"

# ============================================================
# Feature selection for dependence plot
# Option 1: give exact feature name
# Option 2: keep FEATURE_NAME = None and search by terms
# ============================================================

FEATURE_NAME_LIST =[
    "DOXIMITY_DOCNEWS_EMAIL_HEADLINE_VIEW_R_P3",
    "DOXIMITY_DOCPOINT_VIDEO_PLAY_5S_I_P3",
    "DOXIMITY_DOCSPOT_VIDEO_PLAY_5S_I_P3",
    "PULSEPOINT_BANNER_CLICKED_I_P3",
    "DOXIMITY_DOCNEWS_HEADLINE_VIEW_R_P3",
    "MEDSCAPE_PULSEPOINT_BANNER_PULSEPOINT_MEDIA_IMPRESSION_R_P3",
    "MEDSCAPE_3RD_PARTY_EMAIL_EMAIL_DELIVERY_R_P3",
    "DOXIMITY_ANNOUNCEMENT_CLICK_THROUGH_I_P3",
    "MEDSCAPE_3RD_PARTY_EMAIL_EMAIL_OPEN_I_P3",
    "PEERDIRECT_MD_MINUTE_EMAIL_DELIVERED_R_P3",
    "DOXIMITY_DOCNEWS_EMAIL_CONTENT_VIEW_I_P3",
    "DOXIMITY_ANNOUNCEMENT_CLICK_THROUGH_E_P3",
    "DOXIMITY_ANNOUNCEMENT_VIEW_R_P3",
    "MEDSCAPE_BANNER_PULSEPOINT_MEDIA_IMPRESSION_R_P3",
    "PEERDIRECT_MD_MINUTE_WEB_VIEW_R_P3",
    "MEDSCAPE_3RD_PARTY_EMAIL_EMAIL_CLICK_E_P3",
    "DOXIMITY_DOCPOINT_CLICK_THROUGH_E_P3",
    "MEDSCAPE_PULSEPOINT_BANNER_PULSEPOINT_MEDIA_CLICK_I_P3",
    "PEERDIRECT_MDFI_WEB_VIEW_R_P3",
    "DOXIMITY_DOCSPOT_VIEW_R_P3",
    "DOXIMITY_DOCPOINT_VIEW_R_P3",
    "PEERDIRECT_MDFI_EMAIL_DELIVERED_R_P3",
    "SFMC_HQ_EMAIL_DELIVERED_R_P3",
    "MEDSCAPE_BRANDALERT_PAGE_VIEW_R_P3",
    "DIGITAL_DOXIMITY_EXPOSURE_IN_LAST_12_MONTH",
    "PULSEPOINT_BANNER_IMPRESSIONS_R_P3",
    "DIGITAL_PULSEPOINT_EXPOSURE_IN_LAST_12_MONTH"
]


# Number of rows used for the SHAP dependence plot.
# Recommended: 50k first.
# Use None for all rows, but it will be slower.
SHAP_DEPENDENCE_ROWS = None

# Arrow reads this many rows per parquet batch.
ARROW_BATCH_SIZE = 2048

# LightGBM SHAP prediction batch size.
# Keep moderate because pred_contrib returns n_features + 1 values.
PRED_BATCH_SIZE = 512

# CPU threads
NUM_THREADS = psutil.cpu_count(logical=False)

if NUM_THREADS is None:
    NUM_THREADS = 64

NUM_THREADS = min(NUM_THREADS, 64)

SEED = 42

print("DATA_PATH:", DATA_PATH)
print("MODEL_PATH:", MODEL_PATH)
print("FEATURE_MAPPING_PATH:", FEATURE_MAPPING_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("NUM_THREADS:", NUM_THREADS)

In [0]:
# ============================================================
# Helper functions
# ============================================================

def safe_filename(x):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(x))[:180]


def copy_local_to_output(local_path, output_filename=None):
    local_path = str(local_path)

    if output_filename is None:
        output_filename = os.path.basename(local_path)

    final_output_path = f"{OUTPUT_DIR.rstrip('/')}/{output_filename}"

    try:
        dbutils.fs.cp(
            f"file:{local_path}",
            f"dbfs:{final_output_path}",
            True
        )
    except Exception:
        Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        shutil.copyfile(local_path, final_output_path)

    print("Saved:")
    print(final_output_path)

    return final_output_path


def get_selected_row_indices(n_rows, sample_rows=None, seed=42):
    if sample_rows is None or sample_rows >= n_rows:
        return np.arange(n_rows, dtype=np.int64)

    rng = np.random.default_rng(seed)

    return np.sort(
        rng.choice(
            n_rows,
            size=sample_rows,
            replace=False
        ).astype(np.int64)
    )

In [0]:
# ============================================================
# Load saved LightGBM model
# ============================================================

booster = lgb.Booster(model_file=MODEL_PATH)

print("Model loaded successfully.")
print("Number of model features:", booster.num_feature())
print("Number of trees:", booster.num_trees())
print("Current iteration:", booster.current_iteration())

In [0]:
# ============================================================
# Load feature mapping
# ============================================================

feature_mapping_df = pd.read_csv(FEATURE_MAPPING_PATH)

required_cols = {"feature_index", "feature_alias", "feature"}
missing_cols = required_cols - set(feature_mapping_df.columns)

if len(missing_cols) > 0:
    raise ValueError(f"Feature mapping file missing columns: {missing_cols}")

feature_mapping_df = (
    feature_mapping_df
    .sort_values("feature_index")
    .reset_index(drop=True)
)

feature_cols = feature_mapping_df["feature"].tolist()
feature_aliases = feature_mapping_df["feature_alias"].tolist()

n_features = len(feature_cols)

print("Features in mapping:", n_features)
print("First 10 features:")
print(feature_cols[:10])

if booster.num_feature() != n_features:
    raise ValueError(
        f"Model expects {booster.num_feature()} features, "
        f"but mapping has {n_features} features."
    )

In [0]:
import pyarrow.dataset as ds
import pyarrow.compute as pc

dataset = ds.dataset(DATA_PATH, format="parquet")

dataset_table = dataset.to_table()
target_idx = dataset_table.schema.get_field_index(TARGET_COL)
dataset_table = dataset_table.set_column(
    target_idx,
    TARGET_COL,
    pc.fill_null(dataset_table[TARGET_COL], 0)
)
dataset = ds.dataset(dataset_table)

n_rows = dataset.count_rows()
print("Total rows in dataset:", n_rows)

# Columns to load from disk: all features + PROVIDER_ID + target
required_cols = feature_cols + ["PROVIDER_ID", TARGET_COL]

# Pre-compute selected row mask
selected_indices = get_selected_row_indices(
    n_rows=n_rows,
    sample_rows=SHAP_DEPENDENCE_ROWS,
    seed=SEED
)
selected_set = set(selected_indices.tolist())
n_selected = len(selected_indices)
print("Rows selected for SHAP:", n_selected)

# Pre-compute feature index lookup (much faster than .index() in loop)
selected_feature_indices = np.array(
    [feature_cols.index(f) for f in FEATURE_NAME_LIST],
    dtype=np.int64
)
n_selected_features = len(FEATURE_NAME_LIST)

print("Selected features (with indices):")
for f, idx in zip(FEATURE_NAME_LIST, selected_feature_indices):
    print(f"  [{idx}] {f}")

In [0]:
# ============================================================
# Stream batches: compute SHAP per batch, keep only what we need
# ============================================================

import time

# Pre-allocate result arrays for ALL selected rows
provider_ids_arr  = np.empty(n_selected, dtype=object)
y_values_arr      = np.empty(n_selected, dtype=np.float32)
shap_values_arr   = np.empty((n_selected, n_selected_features), dtype=np.float32)
raw_values_arr    = np.empty((n_selected, n_selected_features), dtype=np.float32)

# Scanner with batched Arrow read
scanner = dataset.scanner(
    columns=required_cols,
    batch_size=ARROW_BATCH_SIZE
)

start_time = time.time()
global_row_idx = 0
write_ptr     = 0    # pointer into result arrays

batch_count   = 0

for batch in scanner.to_batches():
    batch_size = batch.num_rows
    batch_global_idx = np.arange(global_row_idx, global_row_idx + batch_size)

    # Filter to selected rows only
    keep_mask = np.isin(batch_global_idx, selected_indices, assume_unique=True)

    if keep_mask.any():
        df_batch = batch.to_pandas()
        df_batch = df_batch.loc[keep_mask].reset_index(drop=True)

        n_keep = len(df_batch)

        # Feature matrix in correct column order
        X_batch = df_batch[feature_cols].values.astype(np.float32, copy=False)

        # SHAP prediction in sub-batches
        shap_full = np.empty(
            (n_keep, booster.num_feature() + 1),
            dtype=np.float32
        )

        sub_start = 0
        while sub_start < n_keep:
            sub_end = min(sub_start + PRED_BATCH_SIZE, n_keep)
            shap_full[sub_start:sub_end] = booster.predict(
                X_batch[sub_start:sub_end],
                pred_contrib=True,
                num_threads=NUM_THREADS
            )
            sub_start = sub_end

        # Keep only the 7 SHAP columns we care about
        shap_keep = shap_full[:, selected_feature_indices]

        # Raw values for the 7 features
        raw_keep = X_batch[:, selected_feature_indices]

        # Write into pre-allocated arrays
        provider_ids_arr[write_ptr : write_ptr + n_keep] = df_batch["PROVIDER_ID"].values
        y_values_arr[write_ptr : write_ptr + n_keep]     = df_batch[TARGET_COL].values.astype(np.float32, copy=False)
        shap_values_arr[write_ptr : write_ptr + n_keep]  = shap_keep
        raw_values_arr[write_ptr : write_ptr + n_keep]   = raw_keep

        write_ptr += n_keep

        # Free memory aggressively
        del X_batch, shap_full, shap_keep, raw_keep, df_batch

    global_row_idx += batch_size
    batch_count += 1

    if batch_count % 50 == 0:
        elapsed = time.time() - start_time
        pct = 100 * global_row_idx / n_rows
        print(f"  Processed {global_row_idx:>10,} / {n_rows:,} rows  ({pct:5.1f}%)  |  {elapsed:6.1f}s")

# Truncate (in case selected_indices had fewer hits than expected)
provider_ids_arr = provider_ids_arr[:write_ptr]
y_values_arr     = y_values_arr[:write_ptr]
shap_values_arr  = shap_values_arr[:write_ptr]
raw_values_arr   = raw_values_arr[:write_ptr]

print("\n===================================================")
print(f"DONE — {write_ptr:,} rows processed")
print(f"Total time: {time.time() - start_time:.1f}s")
print("===================================================")

In [0]:
# ============================================================
# Build final dataframe & save (efficient column construction)
# ============================================================

# Use dict-of-arrays for fast DataFrame construction (no copy)
result_data = {
    "PROVIDER_ID": provider_ids_arr,
    TARGET_COL:    y_values_arr,
}

for i, feat in enumerate(FEATURE_NAME_LIST):
    result_data[f"SHAP_{feat}"]  = shap_values_arr[:, i]
    result_data[f"VALUE_{feat}"] = raw_values_arr[:, i]

result_df = pd.DataFrame(result_data)

print("Final dataframe shape:", result_df.shape)
print("Columns:", list(result_df.columns))
print("\nMemory usage:")
print(result_df.memory_usage(deep=True).sum() / 1024**2, "MB")
print("\nSample:")
print(result_df.head())

# Save locally then copy to volume
local_path = f"{LOCAL_OUTPUT_DIR}/shap_values_per_hcp_new_v7.parquet"
result_df.to_parquet(
    local_path,
    index=False,
    compression="snappy",
    engine="pyarrow"
)

copy_local_to_output(local_path, "shap_values_per_hcp_new_v7.parquet")

In [0]:
# ============================================================
# Summary: mean SHAP by Y variable group
# ============================================================

# Define groups
mask_positive = (result_df[TARGET_COL] > 0) & result_df[TARGET_COL].notna()
mask_zero_null = (result_df[TARGET_COL] == 0) | result_df[TARGET_COL].isna()

n_positive  = int(mask_positive.sum())
n_zero_null = int(mask_zero_null.sum())

print("===================================================")
print("ROW COUNTS BY GROUP")
print("===================================================")
print(f"Y > 0 (prescribers)        : {n_positive:>10,} rows  ({100*n_positive/len(result_df):.2f}%)")
print(f"Y = 0 or NULL (non-writers): {n_zero_null:>10,} rows  ({100*n_zero_null/len(result_df):.2f}%)")
print(f"Total                      : {len(result_df):>10,} rows")
print("===================================================\n")

# Build comparison summary
shap_cols = [f"SHAP_{f}" for f in FEATURE_NAME_LIST]

summary_rows = []
for feat, shap_col in zip(FEATURE_NAME_LIST, shap_cols):
    mean_positive  = result_df.loc[mask_positive,  shap_col].mean()
    mean_zero_null = result_df.loc[mask_zero_null, shap_col].mean()
    abs_diff       = mean_positive - mean_zero_null

    summary_rows.append({
        "feature":            feat,
        "mean_shap_Y_gt_0":   mean_positive,
        "mean_shap_Y_eq_0":   mean_zero_null,
        "diff_(gt0 - eq0)":   abs_diff,
        "n_rows_Y_gt_0":      n_positive,
        "n_rows_Y_eq_0":      n_zero_null,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values("diff_(gt0 - eq0)", key=abs, ascending=False).reset_index(drop=True)

print("MEAN SHAP COMPARISON (Y > 0 vs Y = 0/NULL):")
print("===================================================")
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:>10.4f}"))
print("===================================================")

# Save summary
local_summary = f"{LOCAL_OUTPUT_DIR}/shap_summary_by_y_group_new_v7.csv"
summary_df.to_csv(local_summary, index=False)
copy_local_to_output(local_summary, "shap_summary_by_y_group_new_v7.csv")

In [0]:
import shap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Selected feature columns and corresponding SHAP columns
beeswarm_features = [
    f for f in FEATURE_NAME_LIST
    if f in result_df.columns and f"SHAP_{f}" in result_df.columns
]

missing_features = [
    f for f in FEATURE_NAME_LIST
    if f not in result_df.columns or f"SHAP_{f}" not in result_df.columns
]

if len(missing_features) > 0:
    print("Missing features skipped:")
    for f in missing_features:
        print(f)

print(f"Features used in beeswarm: {len(beeswarm_features)}")

shap_cols = [f"SHAP_{f}" for f in beeswarm_features]

# Feature values for coloring in beeswarm
X_beeswarm = (
    result_df[beeswarm_features]
    .replace([np.inf, -np.inf], np.nan)
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)

# SHAP values
shap_values_beeswarm = (
    result_df[shap_cols]
    .replace([np.inf, -np.inf], np.nan)
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .values
)

# Plot beeswarm
plt.figure(figsize=(14, max(8, 0.45 * len(beeswarm_features))))

shap.summary_plot(
    shap_values_beeswarm,
    X_beeswarm,
    feature_names=beeswarm_features,
    plot_type="dot",
    max_display=len(beeswarm_features),
    show=False
)

plt.title("SHAP Beeswarm Plot - Selected Features", fontsize=14)
plt.tight_layout()

# Save locally
local_beeswarm_path = f"{LOCAL_OUTPUT_DIR}/shap_beeswarm_selected_features_new_v7.png"
plt.savefig(local_beeswarm_path, dpi=300, bbox_inches="tight")

plt.show()
plt.close()

# Copy to output volume
copy_local_to_output(local_beeswarm_path, "shap_beeswarm_selected_features_new_v7.png")

print("Saved beeswarm plot to:", local_beeswarm_path)